In [4]:
# imports 

import os
import requests
from dotenv import load_dotenv
from bs4 import BeautifulSoup
from IPython.display import Markdown, display
from openai import OpenAI

In [5]:
# Here it is - see the base_url

openai = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')


In [16]:
# To give you a preview -- calling OpenAI with these messages is this easy. Any problems, head over to the Troubleshooting notebook.

message = "Hello, Llama! This is my first ever message to you! Hi!"
llm_model = "deepseek-r1:8b" # replace with your model name
response = openai.chat.completions.create(model=llm_model, messages=[{"role":"user", "content":message}])
print(response.choices[0].message.content)

Hey there! 😊 Welcome—sounds like you're already starting our conversation with such warm vibes! Feel free to poke, question, or just hang out, any of that is totally fine. So… anything you'd like to chat about, ask, or share your first-of-their-kind message?


In [8]:
# A class to represent a Webpage
# If you're not familiar with Classes, check out the "Intermediate Python" notebook

# Some websites need you to use proper headers when fetching them:
headers = {
 "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}

class Website:

    def __init__(self, url):
        """
        Create this Website object from the given url using the BeautifulSoup library
        """
        self.url = url
        response = requests.get(url, headers=headers)
        soup = BeautifulSoup(response.content, 'html.parser')
        self.title = soup.title.string if soup.title else "No title found"
        for irrelevant in soup.body(["script", "style", "img", "input"]):
            irrelevant.decompose()
        self.text = soup.body.get_text(separator="\n", strip=True)

In [ ]:
# Let's try one out. Change the website and add print statements to follow along.

ed = Website("https://edwarddonner.com")
print(ed.title)
print(ed.text)

In [10]:
system_prompt = "You are an assistant that analyzes the contents of a website \
and provides a short summary, ignoring text that might be navigation related. \
Respond in markdown."



In [11]:
def user_prompt_for(website):
    user_prompt = f"You are looking at a website titled {website.title}"
    user_prompt += "\nThe contents of this website is as follows; \
please provide a short summary of this website in markdown. \
If it includes news or announcements, then summarize these too.\n\n"
    user_prompt += website.text
    return user_prompt
print(user_prompt_for(ed))

You are looking at a website titled Home - Edward Donner
The contents of this website is as follows; please provide a short summary of this website in markdown. If it includes news or announcements, then summarize these too.

Home
AI Curriculum
Proficient AI Engineer
Connect Four
Outsmart
An arena that pits LLMs against each other in a battle of diplomacy and deviousness
About
Posts
Well, hi there.
I’m Ed. I like writing code and experimenting with LLMs, and hopefully you’re here because you do too. I also enjoy amateur electronic music production (
very
amateur) and losing myself in
Hacker News
, nodding my head sagely to things I only half understand.
I’m the co-founder and CTO of
Nebula.io
. We’re applying AI to a field where it can make a massive, positive impact: helping people discover their potential and pursue their reason for being. I’m previously the founder and CEO of AI startup untapt,
acquired in 2021
.
I will happily drone on for hours about LLMs to anyone in my vicinity.

In [12]:
def messages_for(website):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(website)}
    ]

In [13]:
def summarize(url):
    website = Website(url)
    response = openai.chat.completions.create(
        model = llm_model,
        messages = messages_for(website)
    )
    return response.choices[0].message.content

In [14]:
def display_summary(url):
    summary = summarize(url)
    display(Markdown(summary))

In [17]:
display_summary("https://cnn.com")

```markdown
# CNN Website Preview

CNN presents a comprehensive news portal covering multiple topics. Key sections include:

## Main News Categories
- **Politics**: US Politics, International, Elections (e.g., 2026 focus), Fact-checking.
- **World**: Featured stories include Hungary's Orbán defeat.
- **Business**: Tech, markets, investing.
- **Health**: Articles on conditions and treatments.
- **Entertainment**: Celebrity news, awards.
- **Sports**: Top stories like McIlroy winning the Masters.
- **Science/Technology**: Space news (Artemis II), innovation.
- **Lifestyle**: Health, culture.

## Featured Stories
- Hungary elects its first female prime minister following Orbán's defeat (multiple articles/videos).
- US-Iran tensions: Diplomatic efforts, military actions, expert analysis.
- Tech industry news (e.g., IBM settlement).
- Sports highlights (Masters tournament, Olympic updates).
- Climate and environmental issues.

## Video & Multimedia
- Extensive coverage of breaking news via short videos (e.g., heists, weather events, political reactions).

## Site Navigation
The site features tabs, menus, and subscription options to navigate various content areas and sign in.

## Other Features
- Interactive tools (calculators, news briefings like '5 Things').
- App download prompts in various languages.
- Subscription, account settings, and news alerts options.
```